# 64장 · LangGraph interrupt와 승인 후 재개

승인이 필요한 그래프는 외부 작업 직전에 실행을 멈추고 상태를 저장합니다. 승인 또는 거절을 전달하면 그 실행을 이어갑니다.

과제 등록 오류 문의를 접수하기 직전에 LangGraph의 interrupt로 실행을 멈춥니다. 같은 실행을 승인 또는 거절로 재개한 뒤 모의 쓰기 도구의 호출 횟수를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## interrupt()로 승인 대기

아래 그래프의 노드는 interrupt에서 실행을 멈춥니다. 체크포인터가 대기 중인 상태를 보관합니다. 여기서는 수업용 메모리를 쓰므로 커널을 다시 시작하면 기록이 사라집니다.

**승인 대기와 실행 재개**　게이트 앞에서 멈춘 요청은 승인 뒤 기록부로 이동합니다. interrupt() 직후와 재개 후에 등록 건수가 어떻게 달라지는지 비교하세요.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter64/index.html#concept-clip)


### 셀 01 · 승인 노드와 체크포인터 구성

노드에서 부수 효과가 생기는 줄은 interrupt 다음에만 둡니다. 같은 승인 번호로 두 번 저장하지 않도록 메모리 사전에서 확인합니다.

처음 interrupt에 닿으면 제안만 보이고 문의 등록은 0건입니다. 같은 작업을 재개하면 이 줄이 승인 또는 거절 값을 반환합니다. 승인 경로만 등록 줄을 지나므로 거절 뒤에는 등록 건수가 0으로 남습니다.


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class Request(TypedDict, total=False):
    approval_id: str
    text: str
    status: str

inquiries = {}
def approve(state: Request):
    decision = interrupt({"approval_id": state["approval_id"], "inquiry_text": state["text"]})
    if decision != "approve":
        return {"status": "declined"}
    inquiries.setdefault(state["approval_id"], state["text"])
    return {"status": "completed"}

builder = StateGraph(Request)
builder.add_node("confirm", approve)
builder.add_edge(START, "confirm")
builder.add_edge("confirm", END)
graph = builder.compile(checkpointer=InMemorySaver())

실행 순서를 연결했습니다. 아직 invoke하지 않았으므로 대기 중인 요청도, 등록된 문의도 없습니다.


### 문의를 보여 주는 시점과 등록하는 시점

<p>문의 내용은 먼저 상태에 담깁니다. inquiries에 등록되는 줄은 interrupt 뒤에 있습니다. 학생이 문의 내용을 볼 수 있어도 아직 등록 건수는 0일 수 있습니다.</p>


### 셀 02 · interrupt()의 대기 상태

대기 응답이 보여 주는 본문과 실제로 기록된 문의 건수를 확인하세요.


In [2]:
config = {"configurable": {"thread_id": "chat-a-demo"}}
pending = graph.invoke({"approval_id": "A1", "text": "과제 등록 오류를 확인해 주세요."}, config)
print(pending["__interrupt__"][0].value["inquiry_text"])
print(len(inquiries))

과제 등록 오류를 확인해 주세요.
0


문의 본문 다음에 0이 나옵니다. 승인 화면에 문구를 띄울 수 있지만 등록 함수는 아직 호출되지 않았습니다.

요청 내용이 화면에 나타난 시점의 실행 횟수는 0입니다. 승인 대기 중에 이미 쓰기 작업을 해 버리면 이후의 거절은 효력이 없습니다. 승인으로 재개한 경로와 거절한 경로에서 실제 실행 횟수를 각각 비교해야 합니다.


### 사람에게 보이는 승인 대상

<p>interrupt는 approval_id와 inquiry_text를 담아 호출자에게 돌려줍니다. 호출자는 __interrupt__의 value에서 문구를 읽고 이 문의를 등록할지 판단합니다.</p>


### confirm에서 멈춘 실행

<p>그래프에는 START → confirm → END가 등록되어 있지만 첫 invoke는 confirm 안에서 중단됩니다. InMemorySaver가 대기 상태를 보관하고, chat-a-demo는 그 상태를 다시 찾을 때 쓰는 thread_id입니다.</p>

<h3>A1 승인과 A2 거절의 결과 읽기</h3>

<p>chat-a-demo에서 A1을 보낸 첫 invoke는 문의 문구를 돌려주고 멈춥니다. 등록은 0건입니다. 같은 config에 Command(resume="approve")를 전달하면 confirm 노드가 재실행되고 A1이 저장됩니다. 별도의 chat-decline에서 A2를 거절하면 declined가 반환되며 A2는 inquiries에 없습니다.</p>


## Command(resume=…)와 체크포인트

재개하려면 같은 thread_id로 체크포인트를 찾아야 합니다. 서비스에서는 여기에 소유자, 만료 시각, 승인 당시 인자의 해시도 대조합니다. 여기서는 로컬 그래프의 멈춤과 재개에 집중합니다.

<!-- lesson-visual practice-space -->
**승인 여부를 바꿔 같은 문의의 등록 건수 비교**

각 선택은 비어 있는 문의 저장소로 시작하는 독립 실행입니다. 같은 체크포인트의 A1에 전달할 결정을 바꿔 보세요.

대기와 거절은 등록 0건입니다. 승인으로 재개한 실행에서만 completed가 되고 문의가 1건 생깁니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter64/#practice-space)
<!-- /lesson-visual practice-space -->


### 같은 confirm을 다시 실행한 뒤 A1 등록

같은 thread_id로 Command를 보내면 confirm 안의 interrupt가 결정을 반환합니다. 등록은 그 뒤에 한 번 일어납니다.

1. 승인 대기 · thread_id = chat-a-demo
2. confirm 재실행 · 재개 입력 = Command(resume="approve")
3. interrupt 반환 · decision = approve
4. 문의 등록 · inquiries의 새 키 = A1
5. END · status · 총 건수 = completed · 1


### 셀 03 · Command(resume=…)로 승인 전달

기존 대기 요청에 Command를 보냅니다. 같은 질문을 새로 시작하지 않습니다.


In [3]:
done = graph.invoke(Command(resume="approve"), config)
print(done["status"], len(inquiries))

completed 1


completed 1입니다. 같은 승인 번호로 문의 한 건이 생겼습니다.


### 같은 스레드에서 결정을 전달하기

<p>Command(resume="approve")를 처음과 같은 config로 전달합니다. 중단됐던 confirm 노드는 처음부터 다시 실행되고 interrupt 자리는 이번에 받은 결정을 돌려줍니다. 등록 코드를 그 뒤에 두면 대기 시점에 부작용이 생기지 않습니다.</p>


### 같은 승인 ID가 가리키는 항목

<p>inquiries.setdefault는 해당 ID가 없을 때만 새 값을 넣습니다. 이 예제의 중복 방지는 현재 커널의 딕셔너리 안에서 같은 키를 다시 쓸 때의 동작입니다.</p>


### 과제 04 · 거절 분기의 종료 상태

별도의 대기 요청 A2를 만들고 decline을 보내세요. 기록 목록에 A2가 남는지 확인하세요.


In [4]:
# 새 thread_id로 대기를 만들고 거절한 뒤 건수를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>other = {&#x27;configurable&#x27;: {&#x27;thread_id&#x27;: &#x27;chat-decline&#x27;}}
graph.invoke({&#x27;approval_id&#x27;: &#x27;A2&#x27;, &#x27;text&#x27;: &#x27;추가 문의&#x27;}, other)
refused = graph.invoke(Command(resume=&#x27;decline&#x27;), other)
print(refused[&#x27;status&#x27;], &#x27;A2&#x27; in inquiries)</code></pre>
<p>실행 결과</p><pre><code>declined False</code></pre>
<p>declined False가 보입니다. 같은 자료로 외부 문의가 실제 전송됐다고 말할 수는 없습니다.</p>
</details>


### 승인과 거절에서 달라지는 쓰기 횟수

<p>approve 이외의 결정을 받으면 함수는 declined를 바로 반환합니다. 새 문의를 거절했을 때 건수는 거절 직전과 같습니다. 승인 실험을 먼저 했다면 기존 한 건까지 사라지는 것은 아닙니다.</p>


## 돌아보기

대기 전에 외부 등록 코드를 놓으면 재개 시 노드가 다시 실행될 때 문제가 생깁니다. 노드에서 어떤 줄이 다시 평가될 수 있는지 짚어 보세요.

### 대기와 재개의 시간 순서

**읽기와 제안**　동일 Q03을 읽고 원래 사용자·인자·만료를 묶습니다.

```
등록 0
```

**interrupt**　제안 객체를 보여 주고 checkpoint에 대기 상태를 남깁니다.

```
pending_approval / result=null
```

**결정 전달**　같은 요청·승인 ID·해시와 신뢰된 context로 Command를 보냅니다.

```
approve 또는 decline
```

**노드 재실행**　노드 앞부분부터 다시 실행하며 소유자·대상·만료를 확인합니다.

```
검사 후 write 또는 종료
```

**효과와 결과**　승인된 한 건만 등록하고 receipt를 A1과 분리합니다.

```
승인 A1 → 등록 1건
거절 A2 → 추가 등록 0건
```
